In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import polars as pl # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Построение прогнозной модели рыночной цены автомобиля на основе его технических параметров
# Введение

## Задача

На основе параметров автомобиля оценить эффективность стоимости продажи, используя алгоритмы машинного обучения.

## Модель решения задачи
Допустим $A$ фактическая цена продажи указанную в объявлении, $B$ опорная цена, $\delta$ = $A/B-1$
Разобъем сделки на 5 категорий (классов):

* подозрительные => $0 < \delta \text{ И } |\delta|\ge0.20$;
* выгодные => $0 < \delta \text{ и } |\delta| \ge 0.10$;
* норма => $(0 \le \delta \text{ или } 0 > \delta) \text{ и } |\delta|< 0.10$;
* завышено => $0 > \delta \text{ и } |\delta| \ge 0.10$;
* сильно завышено => $0 > \delta \text{ И } |\delta|\ge0.20$;

Опорная цена B определяется с помшью регессии, в качестве моделм предсказывающей цену предполагается использовать бустинговый метод Random Forest, так как предполагается что в данных присутствуют сложные нелинейные зависимости, а объем данных достаточнен для того чтобы избежать переобучения модели.

## Описание набора данных

Набор CarSales представляет из себя табличный файл Comma Separated Values. В нем собраны результаты парсинга сайтов с объявлениями о продаже автомобилей в Российской Федерации. Данные собирались в 2023 году. Размещен в открытый доступ на платформе Kaggle. Содержит 1 294 757 экземпляров, каждый из которых имеет по 18 признаков. Экземпляр содержит как численные так и категориальные признаки.

## Стек технологий

1) Polars - фреймворк для обработки данных, альтернатива Pandas;
2) sklearn - библиотека содержащая модели класического машинного обучения;
3) seaborn и matplotlib - библиотеки для создания графиков;
4) numpy - библиотека для линейной алгебры.


# Анализ и подготовка данных

## Чтение данных

In [ ]:
df = pl.read_csv("/kaggle/input/datasets/ekibee/car-sales-information/all_regions.csv")
print(f"Размер датасета (строки, колонки): {df.shape}")

## Обзор типов данных признаков

Polars - крайне строгий фреймворк в отношении типов данных

In [ ]:
for col_name, dtype in df.schema.items():
    print(f"{col_name}: {dtype}")

In [ ]:
print("\n--- Первые 5 строк ---")
print(df.head())

In [ ]:
print("\n--- Количество пропусков ---")
print(df.null_count())

In [ ]:
print("\n--- Базовые статистики ---")
print(df.describe())

### Подготовка к очистке данных

In [ ]:
# 1. Исходный размер
print(f"1. Исходный размер: {df.shape}")

# 2. Проверяем потери на фильтре цен
df_price_filtered = df.filter(
    (pl.col("price") > 50_000) & 
    (pl.col("price") < 50_000_000)
)
print(f"2. После фильтра цены: {df_price_filtered.shape} (Потеряно: {df.shape[0] - df_price_filtered.shape[0]})")

# 3. Смотрим, сколько изначальных пропусков было в критичных колонках
print("\n3. Изначальные пропуски (nulls) в числах:")
print(df_price_filtered.select([
    pl.col("year").null_count(), 
    pl.col("mileage").null_count(), 
    pl.col("power").null_count()
]))

# 4. Пробуем сделать cast и смотрим, не добавило ли это новых nulls
df_casted = df_price_filtered.with_columns([
    pl.col("year").cast(pl.Int32, strict=False),
    pl.col("mileage").cast(pl.Int32, strict=False),
    pl.col("power").cast(pl.Int32, strict=False)
])
print("\n4. Пропуски после приведения типов (cast):")
print(df_casted.select([
    pl.col("year").null_count(), 
    pl.col("mileage").null_count(), 
    pl.col("power").null_count()
]))

Фильтр по цене сработал хорошо, мы убрали около 0,7% выбросов, однако специфика данных такова, что большая часть пропусков находятся в столбцах с пробегом, годом выпуска и мощьностью. Удалять больше половины данных нельзя.

Предлагаемое решение - заполнить пропуски медианными значениями основываясь на похожих объявлениях.

## Очистка данных

In [ ]:
df_clean = (
    df
    .filter(
        (pl.col("price") > 50_000) & 
        (pl.col("price") < 50_000_000)
    )
    .drop(["link", "description", "date", "parse_date"])
    
    .with_columns([
        pl.col("color").fill_null("unknown"),
        pl.col("vehicleConfiguration").fill_null("unknown"),
        
        pl.col("engineDisplacement")
        .fill_null(pl.col("engineDisplacement").drop_nulls().mode().first().over(["brand", "name", "bodyType"]))
        .fill_null(pl.col("engineDisplacement").drop_nulls().mode().first().over(["brand", "name"]))
        .fill_null("unknown"),
        
        pl.col("engineName")
        .fill_null(pl.col("engineName").drop_nulls().mode().first().over(["brand", "name", "bodyType"]))
        .fill_null(pl.col("engineName").drop_nulls().mode().first().over(["brand", "name"]))
        .fill_null("unknown"),
    ])
    
    .with_columns([
        pl.col("year")
        .fill_null(pl.col("year").median().over(["brand", "name", "bodyType", "engineDisplacement", "engineName", "fuelType"]))
        .fill_null(pl.col("year").median().over(["brand", "name", "bodyType", "engineDisplacement"]))
        .fill_null(pl.col("year").median().over(["brand", "name", "bodyType"]))
        .fill_null(pl.col("year").median().over(["brand", "name"]))
        .fill_null(pl.col("year").median()),

        pl.col("mileage")
        .fill_null(pl.col("mileage").median().over(["brand", "name", "bodyType", "engineDisplacement", "engineName", "fuelType"]))
        .fill_null(pl.col("mileage").median().over(["brand", "name", "bodyType", "engineDisplacement"]))  
        .fill_null(pl.col("mileage").median().over(["brand", "name", "bodyType"]))
        .fill_null(pl.col("mileage").median().over(["brand", "name"]))
        .fill_null(pl.col("mileage").median()),

        pl.col("power")
        .fill_null(pl.col("power").median().over(["brand", "name", "bodyType", "engineDisplacement","engineName", "fuelType", "transmission"]))
        .fill_null(pl.col("power").median().over(["brand", "name", "bodyType", "engineDisplacement"]))
        .fill_null(pl.col("power").median().over(["brand", "name", "bodyType"]))
        .fill_null(pl.col("power").median().over(["brand", "name"]))
        .fill_null(pl.col("power").median()),
    ])
    
    .with_columns([
        pl.col("year").cast(pl.Int32, strict=False),
        pl.col("mileage").cast(pl.Int32, strict=False),
        pl.col("power").cast(pl.Int32, strict=False)
    ])
    .drop_nulls(subset=["power", "mileage", "year"])
)

print(f"Размер после очистки: {df_clean.shape}")
print(f"Пропусков осталось:\n{df_clean.null_count().sum()}") 

## Анализ зависимостей

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Анализ корреляций Пирсона для численных признаков
num_cols = ["price", "year", "mileage", "power"]
df_numeric = df_clean.select(num_cols).to_pandas()

# Строим матрицу корреляций
corr_matrix = df_numeric.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Матрица корреляций численных признаков")
plt.tight_layout()
plt.show()

# 2. Визуализация распределения целевой переменной (Цены)
plt.figure(figsize=(12, 5))

# Оригинальная цена
plt.subplot(1, 2, 1)
sns.histplot(df_numeric['price'], bins=50, kde=True, color='blue')
plt.title("Распределение цены (Original)")
plt.xlabel("Цена (руб.)")
plt.ylabel("Частота")
# Отключаем научный формат оси X для удобства чтения
plt.ticklabel_format(style='plain', axis='x') 
plt.xticks(rotation=45)

# Логарифмированная цена
# np.log1p берет натуральный логарифм (log(1+x)), что безопасно для нулей
plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df_numeric['price']), bins=50, kde=True, color='green')
plt.title("Распределение цены (Log Transform)")
plt.xlabel("Log(Цена)")
plt.ylabel("Частота")

plt.tight_layout()
plt.show()

Матрица корреляций Пирсона для числовых признаков хорошо показывает, существующие взаимосвязи в данных, которые выглядят достаточно логично, целевая величина положительно коррелирует с мощностью и годом (чем новее и мощнее автомобиль тем он дороже) и отрицательно коррелирует с пробегом, чем больше пробег тем дешевле автомобиль. Ожидается, что эти признаки будут одними из самых значимых при предсказании моделью цены.

Графики распределений

Распределение цены в рублях показывает пик в районе 1 миллиона рублей далее график идет на спад, и мы видим длинный "хвост" до 50 млн., что не является оптимальным для обучения. Распределение меньшего масштаба (логарифмированная цена) показывает очивидное скопление в центре и снижение частоты на концах, этот график ближе к нормальному распределение, поэтому для лучшего преедсказания будет использована именно логарифмированная цена.

In [ ]:
categorical_cols = ["brand", "name", "bodyType", "color", "fuelType", "transmission", "vehicleConfiguration", "engineName", "engineDisplacement", "location"]
print(df_clean.select([pl.col(c).n_unique().alias(c) for c in categorical_cols]))

Блок выше показывает что у нас есть большое множество уникальных категориальных признаков. Работа с библиотекой sklearn подразумевает, что даже для деревьев решений, которые умеют работать с категориальными признаками необходимо приводить их в числовые, поэтому необходимо использовать кодирование признаков.

Классический One-Hot Encoding не подойдет так как получится  > 5000 новых признаков каждый из которых длиной 1,28 млн. значений, это увеличит матрицу до десятков гигабайт и не позволит корректно работать с ней.

Label Encoding так же имеет существенный минус в контексте нашей задачи. Предположим, мы закодировали Audi как 1, BMW как 2 и Лада как 3, тогда для тех же деревьев решений это будет выглядеть так что Лада значит в 3 раза больше чем Audi, что является нежелательной зависимостью.

Лучше всего для задачи подойдет Binary Encoding, его минус заключается в том что ухудшается интерпретируемость, но при этом у него отсутсвуют минусы первых двух кодировок, так что его использование вполне обосновано

## Кодирование категориальных признаков с помощью Binary Encoding

In [ ]:
import math
import polars as pl

categorical_cols = ["brand", "name", "bodyType", "color", "fuelType", "transmission", "vehicleConfiguration", "engineName", "engineDisplacement", "location"]

df_numeric_cats = df_clean.with_columns([
    pl.col(c).rank("dense").cast(pl.UInt32).alias(c) 
    for c in categorical_cols
])

binary_exprs = []

for c in categorical_cols:
    max_id = df_numeric_cats.select(pl.col(c).max()).item()
    num_bits = math.ceil(math.log2(max_id)) if int(max_id) > 1 else 1
    
    for i in range(num_bits):
        bit_expr = (
            ((pl.col(c) & (1 << i)) > 0)
            .cast(pl.Int8) 
            .alias(f"{c}_bin_{i}")
        )
        binary_exprs.append(bit_expr)

df_encoded = (
    df_numeric_cats
    .with_columns(binary_exprs)
    .drop(categorical_cols))

print(f"Размер датасета после Binary Encoding: {df_encoded.shape}")
print("\n--- Пример бинарных колонок для топлива ---")
print(df_encoded.select(pl.col("^fuelType_bin_.*$")).head())

# Обучение опорной модели

## Подготовка выборок и обучение

Для обоснования использования продвинутого алгоритма Random Forest, приведено обучение вариации линейной регрессии Ridge Regression, если она покажет плохой результат значит данные не разделяются с помощью гиперплоскости, а значит использование бэггинга над деревьями решений является оправданным. 

Так же было принято решение добавить признак характеризующий средний пробег за год, это позволит оценить насколько активно машина использовалась.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import pandas as pd
import numpy as np
import polars as pl

CURRENT_YEAR = 2026
# Добавляем умные признаки до разделения
df_encoded = (
    df_encoded
    .with_columns([
        pl.max_horizontal(1, CURRENT_YEAR - pl.col("year")).alias("car_age")
    ])
    .with_columns([
        (pl.col("mileage") / pl.col("car_age")).alias("mileage_per_year")
    ])
)

print("Подготовка X и y...")
y_log = np.log1p(df_encoded.select("price").to_numpy().ravel())
X = df_encoded.drop("price").to_pandas()
X = X.fillna(0) # Фикс нулей для бинарного энкодера

X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
y_test_real = np.expm1(y_test_log)

print("--- Ridge Regression ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_log)

preds_ridge_log = ridge.predict(X_test_scaled)
preds_ridge_real = np.expm1(preds_ridge_log)
print(f"MAPE Ridge: {mean_absolute_percentage_error(y_test_real, preds_ridge_real) * 100:.2f}%")


print("\n--- Опорная модель: Случайный лес ---")
print("Обучение...")
rf = RandomForestRegressor(
    n_estimators=120,
    max_depth=35,
    max_leaf_nodes=100000,
    min_samples_split=5,     
    min_samples_leaf=2,      
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train_log)
print("Обучение завершено!")

preds_rf_log = rf.predict(X_test)
preds_rf_real = np.expm1(preds_rf_log)

# подсчет метрик (сравниваем реальные рубли с предсказанными рублями)
mae = mean_absolute_error(y_test_real, preds_rf_real)
mape = mean_absolute_percentage_error(y_test_real, preds_rf_real) * 100

print(f"Средняя ошибка (MAE): {mae:,.0f} руб.")
print(f"Средняя ошибка в процентах (MAPE): {mape:.2f}%")

# --- Бизнес-логика ---
results = pd.DataFrame({
    'Реальная цена': y_test_real,
    'Предсказание': preds_rf_real
})
results['Разница_%'] = ((results['Реальная цена'] - results['Предсказание']) / results['Предсказание']) * 100

def evaluate_deal(diff):
    if diff > 20: return "Сильно завышено"
    elif 10 < diff <= 20: return "Завышено"
    elif -10 <= diff <= 10: return "Норма"
    elif -20 <= diff < -10: return "Выгодно"
    else: return "Подозрительно"

results['Оценка'] = results['Разница_%'].apply(evaluate_deal)
print("\n--- Распределение оценки объявлений ---")
print(results['Оценка'].value_counts())

Модель показала что в предсказании цены автомобила она в среднем ошибается на 15,45% это является хорошим результатом, так как в обучении мы не учитываем сезонность, срочность продажи и внешний вид, а так же описание автомобиля от продавца. Ошибка регрессии в ~37% подтвержадает то что использование Random Forest оправдано

## Оценка адекватности модели

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

importances = rf.feature_importances_
cols = X_train.columns.tolist()

df_imp = pl.DataFrame({
    "feature": cols,
    "importance": importances
})

df_imp_agg = (
    df_imp
    .with_columns(
        pl.col("feature").str.replace(r"_bin_\d+$", "").alias("base_feature")
    )
    .group_by("base_feature")
    .agg(pl.col("importance").sum())
    .sort("importance", descending=True)
)

print("\n--- Топ-5 факторов, влияющих на цену ---")
print(df_imp_agg.head())

pdf_imp = df_imp_agg.to_pandas()

plt.figure(figsize=(10, 6))
sns.barplot(
    data=pdf_imp, 
    x="importance", 
    y="base_feature", 
    hue="base_feature",
    palette="viridis",
    legend=False
)
plt.title("Что диктует цену на авторынке? (Feature Importances)")
plt.xlabel("Доля важности (от 0 до 1.0)")
plt.ylabel("Характеристика")
plt.tight_layout()
plt.show()

Как и предполагалось год, мощьность, пробег и возраст оказались наиболее значимыми признаками по которым модель принимала решения. Это значит, что модель ведет себя предсказуемо.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


sns.set_theme(style="whitegrid", palette="muted")

# Берем случайную подвыборку, чтобы графики строились быстро и не превращались в сплошное пятно
sample_results = results.sample(n=10000, random_state=42).copy()

# Создаем большое полотно для дашборда
fig = plt.figure(figsize=(18, 12))

# --- График 1: Реальная цена vs Предсказанная (Scatter plot) ---
ax1 = fig.add_subplot(2, 2, 1)
sns.scatterplot(
    x='Реальная цена', 
    y='Предсказание', 
    data=sample_results, 
    alpha=0.5, 
    color='#2ecc71',
    edgecolor=None,
    ax=ax1
)
# Рисуем идеальную линию предсказания (где Реальная = Предсказанной)
max_val = 5_000_000 
ax1.plot([0, max_val], [0, max_val], color='#e74c3c', linestyle='--', linewidth=2)
ax1.set_xlim(0, max_val)
ax1.set_ylim(0, max_val)
ax1.ticklabel_format(style='plain', axis='both')
ax1.set_title('Реальная цена vs Предсказанная (до 5 млн руб.)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Реальная цена (руб.)')
ax1.set_ylabel('Предсказание модели (руб.)')

# --- График 2: Распределение процентной ошибки ---
ax2 = fig.add_subplot(2, 2, 2)
error_dist = sample_results[sample_results['Разница_%'].between(-50, 50)]
sns.histplot(error_dist['Разница_%'], bins=50, kde=True, color='#3498db', ax=ax2)
ax2.axvline(0, color='#e74c3c', linestyle='--', linewidth=2)
ax2.set_title('Распределение ошибки модели (%)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Ошибка: (Реальная - Предсказание) в %')
ax2.set_ylabel('Количество объявлений')

# --- График 3: Разброс ошибок по ценовым сегментам (Boxplot) ---
ax3 = fig.add_subplot(2, 1, 2)

bins = [0, 500_000, 1_000_000, 3_000_000, np.inf]
labels = ['Бюджет (<500к)', 'Средний (500к - 1М)', 'Комфорт (1М - 3М)', 'Премиум (>3М)']
sample_results['Сегмент'] = pd.cut(sample_results['Реальная цена'], bins=bins, labels=labels)

sns.boxplot(
    x='Сегмент', 
    y='Разница_%', 
    data=sample_results[sample_results['Разница_%'].between(-60, 60)], 
    palette='Set2',
    ax=ax3
)
ax3.axhline(0, color='#e74c3c', linestyle='--', linewidth=2)
ax3.set_title('Разброс ошибок (MAPE) по ценовым сегментам', fontsize=14, fontweight='bold')
ax3.set_xlabel('Ценовой сегмент')
ax3.set_ylabel('Ошибка (%)')

plt.tight_layout()
plt.show()

Эти графики наглядно демонстрируют то, насколько хорошо отработал случайный лес. На первом графике видно плотное скопление точек около красной диагонали, это означает что модель хорошо выучила зависимости в данных и правильно их группирует. Второй график показывает что распределение ошибки предсказания близко к нормальному, и ошибки не смещены относительно нуля, а значит модель адекватно оценивает рынок, не делает переоценивания или недооценивания

# Альтернаиива: использование CatBoost

CatBoost (Categorical Boosting) представляет из себя продвинутую библиотеку градиентного бустинга разработанную в Яндекc. Ее главное отличие заложено прямо в названии: она изначально спроектирована для безупречной работы с категориальными (текстовыми) признаками "из коробки".

## Бустинг против Бэггинга

* Бэггинг (Случайный лес): Строит сотни глубоких деревьев независимо друг от друга и параллельно. Каждое дерево пытается предсказать итоговую цену самостоятельно, а в конце берется усредненное значение. Это хорошо защищает от переобучения, но алгоритм ограничен потолком точности индивидуальных деревьев.

* Бустинг (CatBoost): Строит деревья последовательно. Первое дерево делает грубое предсказание цены. Второе дерево пытается предсказать не саму цену, а ошибку первого дерева. Третье — ошибку второго, и так далее. Это направленный процесс самосовершенствования.

Особенности делающие алгоритм привлекательным для решения задачи:

1) Нативная работа с текстом: Алгоритм под капотом использует сложные математические расчеты для перевода строк в числа. Он умеет находить связи в сырых названиях "Toyota" или "Camry" лучше, чем большинство ручных кодировок.

2) Симметричные деревья (Oblivious Trees): В отличие от Леса или LightGBM, CatBoost строит идеально симметричные деревья (условия разбиения на одном уровне дерева всегда одинаковые).

3) Борьба с утечкой данных (Ordered Boosting): При кодировании категорий CatBoost использует искусственное "время", чтобы для расчета статистики конкретной строки использовать только те данные, которые алгоритм "видел" до нее. Это полностью исключает переобучение на целевой переменной.

Тем не менее CatBoost может показать результат хуже, чем Random Forest в силу своих особенностей, так как алгоритм строит очень большое количество деревьев, вычислительные ресурсы ограничивают их глубину, что может привести к тому что модель не сможет выучить сложные зависимости в данных. 

Протестируем алгоритм на нашем наборе данных.

## Особенности подготовки данных

Так как CatBoost умеет работь с категориями без дополнительной обработки нам не следует использовать df_encoded в котором категории бинарно закодированы, вместо этого используем созданный ранее df_clean с заполнеными пропусками.

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from catboost import CatBoostRegressor

print("1. Feature Engineering на df_clean...")
CURRENT_YEAR = 2026

# Добавляем умные признаки прямо в сырой датасет
df_cb = (
    df_clean
    .with_columns([
        pl.max_horizontal(1, CURRENT_YEAR - pl.col("year")).alias("car_age")
    ])
    .with_columns([
        (pl.col("mileage") / pl.col("car_age")).alias("mileage_per_year")
    ])
)

print("2. Подготовка X и y...")
# Целевая переменная — логарифм цены
y_log = np.log1p(df_cb.select("price").to_numpy().ravel())

# Признаки переводим в Pandas
X = df_cb.drop("price").to_pandas()

# Явно задаем список категориальных колонок
cat_features = [
    "brand", "name", "bodyType", "color", "fuelType", 
    "transmission", "vehicleConfiguration", "engineName", 
    "engineDisplacement", "location"
]

# Важный нюанс: CatBoost не любит, когда в категориальных колонках числа или NaN.
# Принудительно приводим все категории к типу string (текст)
X[cat_features] = X[cat_features].astype(str)

print("3. Разделение на Train / Test...")
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
y_test_real = np.expm1(y_test_log)

print("4. Обучение CatBoost...")
# Инициализируем модель
cb_model = CatBoostRegressor(
    iterations=1300,          # Максимальное количество деревьев (эпох)
    learning_rate=0.1,        # Шаг обучения (скорость)
    depth=5,                  # Глубина симметричного дерева
    cat_features=cat_features,# Передаем список текстов!
    loss_function='RMSE',     # Оптимизируем квадратичную ошибку
    random_seed=42,
    verbose=100               # Печатать лог каждые 100 итераций
)

# Обучаем, передавая eval_set для контроля переобучения
cb_model.fit(
    X_train, y_train_log,
    eval_set=(X_test, y_test_log),
    early_stopping_rounds=50
)
print("Обучение завершено!")

print("\n5. Финальная оценка...")
# Делаем предсказания и возвращаем их из логарифмов в рубли
preds_cb_log = cb_model.predict(X_test)
preds_cb_real = np.expm1(preds_cb_log)

mae = mean_absolute_error(y_test_real, preds_cb_real)
mape = mean_absolute_percentage_error(y_test_real, preds_cb_real) * 100

print(f"--- Метрики CatBoost ---")
print(f"Средняя ошибка (MAE): {mae:,.0f} руб.")
print(f"Средняя ошибка в процентах (MAPE): {mape:.2f}%")

Алгоритм показал результат хуже чем Random Forest, при этом его обучение заняло значительно больше времени чем Случайного леса, поэтому в данном случае принято решение использовать именно Случайный лес в качестве орпорной модели для предсказания.

# Экспорт модели для бэкэнда

In [ ]:
!pip install skl2onnx onnx

In [ ]:
import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

print("1. Подготовка схемы данных для ONNX...")
# Узнаем точное количество признаков (у нас их 85)
# Крайне важно, чтобы массив в Rust строго совпадал с X_train по длине и порядку колонок!
num_features = X_train.shape[1]

# Говорим ONNX: "Ожидай на вход тензор типа float32 размером [N, 85]"
# None означает, что можно предсказывать как 1 машину, так и батч из 1000 машин за раз
initial_type = [('float_input', FloatTensorType([None, num_features]))]

print("2. Конвертация Случайного Леса в ONNX граф...")
# Конвертируем модель (target_opset=12 — самый стабильный стандарт инструкций)
onnx_model = convert_sklearn(
    rf, 
    initial_types=initial_type,
    target_opset=12 
)

print("3. Сохранение модели на диск...")
with open("random_forest_cars.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Готово! Модель random_forest_cars.onnx сохранена.")

# Обязательно сохраняем порядок колонок, чтобы не перепутать их местами в Rust
print("\nПорядок колонок (сохрани его, он понадобится для сборки массива в Rust):")
print(X_train.columns.tolist())

In [ ]:
import json
import polars as pl

print("1. Извлекаем маппинги из df_clean...")

categorical_cols = [
    "brand", "name", "bodyType", "color", "fuelType", 
    "transmission", "vehicleConfiguration", "engineName", 
    "engineDisplacement", "location"
]

mappings = {}

for c in categorical_cols:
    # Воспроизводим ту же самую логику rank("dense"), что была при обучении.
    # Берем уникальные значения, ранжируем их и сортируем для порядка.
    mapping_df = (
        df_clean.select(pl.col(c))
        .unique()
        .drop_nulls() 
        .with_columns(
            pl.col(c).rank("dense").cast(pl.UInt32).alias("id")
        )
        .sort("id")
    )
    
    # mapping_df.rows() возвращает список кортежей, например: [("Бензин", 1), ("Дизель", 2)]
    # Перегоняем это в обычный Python-словарь
    col_map = {str(row[0]): int(row[1]) for row in mapping_df.rows()}
    mappings[c] = col_map

print("2. Сохраняем данные в JSON...")

# Записываем в файл с поддержкой кириллицы (ensure_ascii=False)
with open("category_mappings.json", "w", encoding="utf-8") as f:
    json.dump(mappings, f, ensure_ascii=False, indent=4)

print("Готово! Файл category_mappings.json успешно создан.")

# Выведем небольшой пример, чтобы убедиться, что всё работает как надо
print(f"\nПример маппинга для типа топлива (fuelType):")
print(json.dumps(mappings['fuelType'], ensure_ascii=False, indent=2))